# AI Shipping Document Parser

A document intelligence pipeline that extracts structured data from scanned Bills of Lading using OCR + LLM.

**Pipeline flow:** PDF → pdf2image → Tesseract OCR → GPT-OSS-120B → Structured JSON

This is a simplified demo built on learnings from a production pipeline achieving **97% schema fulfillment**.

The production version included: CNN page classifier, OpenCV preprocessing, bounding box detection, fine-tuned models on Groq, schema validation, and a RAG feedback loop.

---

**Setup:** Add your OpenRouter API key to Colab Secrets as `OPENROUTER_API_KEY`

In [ ]:
# Install dependencies
!apt-get install tesseract-ocr -y -q
!apt-get install poppler-utils -y -q
!pip install pytesseract pillow pdf2image -q

In [ ]:
# Load API key from Colab Secrets (never hardcode API keys)
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError('Please add OPENROUTER_API_KEY to Colab Secrets (key icon in left sidebar)')

print('API key loaded successfully')

In [ ]:
# Upload a Bill of Lading PDF
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print('Uploaded:', file_name)

In [ ]:
# Convert PDF to image and extract raw text with Tesseract OCR
# Note: Raw OCR text is noisy - stamps, handwriting, and poor scan quality affect output
# This is the core challenge the production pipeline was built to handle

from pdf2image import convert_from_path
from PIL import Image
import pytesseract

images = convert_from_path(file_name)
images[0].save('page.png', 'PNG')

img = Image.open('page.png')
text = pytesseract.image_to_string(img)

print('=== RAW OCR OUTPUT (first 1000 chars) ===')
print(text[:1000])

In [ ]:
# Extract structured fields using LLM with schema-aware prompt engineering
# Model: GPT-OSS-120B via OpenRouter (same model family as production)
#
# Key prompt engineering decisions:
# - Schema defined explicitly so model knows exactly what to extract
# - 'ONLY valid JSON' constraint reduces hallucinations
# - Fallback defaults handle missing fields
#
# Known limitations of this simplified version:
# - Without bounding boxes, model can confuse fields that look similar in raw text
# - Noisy OCR can cause missing or inaccurate values
# - No schema validation to catch errors before downstream use

import requests
import json

schema = {
  "documentInfo": {
    "blNumber": "",
    "dateOfIssue": "",
    "placeOfIssue": "",
    "typeOfBL": "NON-NEGOTIABLE if not mentioned",
    "freightPayment": ""
  },
  "vesselInfo": {
    "vesselName": "",
    "voyageNumber": "",
    "portOfLoading": "",
    "portOfDischarge": "",
    "portOfTransshipment": "",
    "serviceType": ""
  },
  "parties": {
    "shipperName": "",
    "shipperAddress": "",
    "consigneeName": "",
    "consigneeAddress": "",
    "notifyPartyName": "",
    "notifyPartyAddress": ""
  },
  "cargoInfo": {
    "containerNumber": "",
    "containerSize": "",
    "sealNumber": "",
    "numberOfPackages": "",
    "packageType": "",
    "grossWeight": "",
    "netWeight": "",
    "volume": "",
    "hsCode": "",
    "cargoDescription": ""
  }
}

prompt = f"""
You are a shipping document expert. Extract the following fields from this Bill of Lading text and return ONLY valid JSON, nothing else.

Text:
{text}

Return ONLY this JSON structure with these exact field names:
{json.dumps(schema, indent=2)}
"""

response = requests.post(
    'https://openrouter.ai/api/v1/chat/completions',
    headers={
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json'
    },
    json={
        'model': 'openai/gpt-oss-120b:free',
        'messages': [
            {'role': 'user', 'content': prompt}
        ]
    }
)

raw_output = response.json()['choices'][0]['message']['content']

try:
    parsed = json.loads(raw_output)
    print('=== EXTRACTED STRUCTURED DATA ===')
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print('Raw output (JSON parsing failed):')
    print(raw_output)